# 🎯 CosyVoice 300M — BookVoice-AI Test
T4 GPU · Türkisch/Deutsch · Voice Cloning

In [ ]:
#@title ⚙️ Schritt 1: Installation (~5 Min)
import subprocess, sys

print('CosyVoice klonen...')
subprocess.run(['git', 'clone', '--recursive', 
    'https://github.com/FunAudioLLM/CosyVoice.git'], 
    capture_output=True)
print('OK!')

sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')

print('Pakete installieren...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '-r', '/content/CosyVoice/requirements.txt'],
    capture_output=True)
print('OK!')

# PyTorch fix
import torch
_orig = torch.load
torch.load = lambda *a, **kw: _orig(*a, **{**kw, 'weights_only': False})

from cosyvoice.cli.cosyvoice import CosyVoice
print('Modell laden (CosyVoice-300M)...')
model = CosyVoice('iic/CosyVoice-300M')
print('FERTIG!')


In [ ]:
#@title 🎤 Schritt 2: Stimme hochladen + transkribieren
from google.colab import files
import torchaudio, whisper, sys
sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')

print('Stimme hochladen (WAV/MP3, max 30 Sek):')
uploaded = files.upload()
voice_file = list(uploaded.keys())[0]

# Stimme laden
speech, sr = torchaudio.load(voice_file)
if sr != 16000:
    speech = torchaudio.functional.resample(speech, sr, 16000)
if speech.shape[0] == 2:
    speech = speech.mean(dim=0, keepdim=True)
speech = speech[:, :16000*28]  # Max 28 Sekunden
print(f'Stimme: {speech.shape[1]/16000:.1f} Sekunden')

# Automatisch transkribieren
print('Transkribiere...')
w_model = whisper.load_model('small')
result = w_model.transcribe(voice_file, language='tr')
prompt_text = result['text'][:200]
print(f'Transkription: {prompt_text}')


In [ ]:
#@title 🎙️ Schritt 3: Text generieren
ziel_text = 'Sûfî düşünceye ve hakikat arayışına alışılmadık bir bakış. Emir Timur ve Yıldırım Bayezid tarihin en büyük savaşlarından birini yaptı.' #@param {type:"string"}

from IPython.display import Audio, display
import torchaudio, sys
sys.path.insert(0, '/content/CosyVoice')

print(f'Generiere: {ziel_text[:50]}...')
for i, result in enumerate(model.inference_zero_shot(
    ziel_text,
    prompt_text,
    speech,
    stream=False
)):
    torchaudio.save(f'/content/cosy_output_{i}.wav', result['tts_speech'], model.sample_rate)
    print(f'✅ Fertig! Anhören:')
    display(Audio(f'/content/cosy_output_{i}.wav'))
